# Preliminary CETI 2024 validation

This notebook tests whether the core A/B kinematic contrast is recovered in an independent preliminary dataset. It starts from a reviewed, buzz-level derived table and never reads raw CETI audio, pressure, inertial, location, deployment-label, demographic, or clock-time data.

The smaller sample does not show a sufficiently clear bimodal structure to make a two-component GMM the validation claim. We therefore use a rank-space extreme-corner comparison: A lies toward short ramp-up/slow motion, B toward long ramp-up/fast motion, and overlapping middle observations remain unclassified.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))

from validation.ceti2024 import (
    classify_corner,
    kinematic_statistics,
    sequence_statistics,
    validate_publication_table,
)

DATA = ROOT / "data/ceti2024/buzz_features.csv"
ceti = pd.read_csv(DATA)
validate_publication_table(ceti)

## Cohort and release scope

The public table contains 336 buzz rows from 13 anonymous unit tokens and 24 anonymous dives. The movement cohort contains 231 buzzes from 10 unit tokens and 19 dives. Pseudonyms are release-local sequential labels with no distributed crosswalk.

In [ ]:
movement = ceti[ceti["has_kinematics"].astype(bool)]
pd.Series({
    "all_buzzes": len(ceti),
    "all_units": ceti["unit_id"].nunique(),
    "all_dives": ceti["dive_id"].nunique(),
    "movement_buzzes": len(movement),
    "movement_units": movement["unit_id"].nunique(),
    "movement_dives": movement["dive_id"].nunique(),
}).rename("n").to_frame()

## Extreme-corner classification and kinematic checks

The classifier uses complete ramp-up/speed observations. Because the two nearest-corner sets can overlap, those overlaps are left unclassified rather than forced into a group. The expected result is 115 A, 106 B, and 115 unclassified buzzes across the full table.

In [ ]:
labels = classify_corner(ceti)
labels.value_counts().reindex(["A", "B", "unclassified"]).rename("n_buzzes").to_frame()

In [ ]:
kinematics = kinematic_statistics(ceti, labels)
kinematics

The interpretation is deliberately positive but bounded: the independent data recover B as faster, higher-roll, straighter and more vertically persistent, and A as slower, deeper and more oscillatory. This supports the core tactic contrast despite limited sample size and without relying on a bimodal GMM or prey echograms. It does not establish every secondary DSWP effect.

## Exploratory sequence summary

The transition table is retained for transparency but should not be pooled into a universal Markov claim. With 19 movement dives and no distributed whale age/sex metadata, sequence and heading-only estimates have modest power.

In [ ]:
transitions = sequence_statistics(ceti, labels)
transitions

## CETI trajectory release gate

The buzz-level table is included. The corresponding numeric reconstructed CETI trajectory bundle is not yet present locally and must be exported and approved by the data stewards before public release. It must follow the DSWP boundary: local relative coordinates only, no raw channels, locations, source labels, or private crosswalk. This notebook does not fabricate that missing asset or imply that the preliminary data are already cleared for publication.

Use `python scripts/run_pipeline.py` to write the labelled table, statistics, manifest, and supported figures under `results/ceti2024/`.